In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import default_risk.config


application_train_df = pd.read_parquet(cfg.RAW_DATA_DIR / "application_train.train-cleaned.parquet")

load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)

with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)

application_train_df.head()


In [ ]:
#minimun preparations necessary to be able to train the model with application_train
Y= application_train_df["TARGET"]
X= application_train_df.drop(columns=["TARGET"])
X.drop(columns=["SK_ID_CURR"],inplace=True)

categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
    X[col] = X[col].astype('category')

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

hiperparams=  {  
    "objective" : 'binary:logistic',
    "random_state" : 42,
    "eval_metric" :"auc",
    "enable_categorical" : True
}

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline")